# Passive Execution Replay

This notebook tests the execution question raised by the Mark Best style live-vs-sim comparison: if the alpha says we should quote passively, do the passive fill assumptions survive event-level market data?

The notebook uses Hibachi BTC/USDT-P top-of-book quotes and trades with millisecond timestamps. It evaluates best-bid/best-ask joining under forecast latency, send latency, cancel latency, queue-ahead assumptions, and maker rebates.

## Execution Model

At each bar close `t`, the strategy waits for forecast and order-send latency before joining the best bid or ask:

$$
t^{live}=t+\ell_{forecast}+\ell_{send}
$$

The order rests until the cancel becomes effective:

$$
t^{dead}=t^{live}+TTL+\ell_{cancel}
$$

We compare three fill assumptions:

1. **Cross fill**: a bid fills when the future ask crosses the bid; an ask fills when the future bid crosses the ask.
2. **Trade-through fill**: a bid fills when sell taker trades print at or below our bid; an ask fills when buy taker trades print at or above our ask.
3. **Queue-adjusted fill**: trade-through volume must first consume assumed queue ahead, then our order quantity.

For a full fill at price `P_f`, markout is measured against the future mid `M_{t+H}`:

$$
m_{bid}=10^4\log\left(\frac{M_{t+H}}{P_f}\right),\qquad
m_{ask}=10^4\log\left(\frac{P_f}{M_{t+H}}\right)
$$

Maker rebate is added in bps per filled side:

$$
EV^{posted}=\mathbb{1}_{fill}\left(m+r_{maker}\right)
$$

A rebate of `-0.025%` as a fee is a positive maker rebate of `2.5 bps` per fill. It helps, but it cannot rescue fills whose adverse markout is much worse than `-2.5 bps`.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

ROOT = Path(os.environ.get("MODL_WS_NORMALIZED_DIR", "/mnt/burner-archive/ws_normalized")).expanduser()
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
RAW_ROOT = Path(os.environ.get("MODL_RAW_DATA_DIR", "data")).expanduser()

BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))
MARKOUT_MINUTES = int(os.environ.get("MODL_EXECUTION_MARKOUT_MINUTES", "30"))
ORDER_QTY = float(os.environ.get("MODL_EXECUTION_ORDER_QTY", "0.01"))
FORECAST_AND_SEND_LATENCY_MS = tuple(int(value) for value in os.environ.get("MODL_EXECUTION_LATENCY_MS", "0,200,500,1000").split(","))
CANCEL_AFTER_MS = tuple(int(value) for value in os.environ.get("MODL_EXECUTION_CANCEL_AFTER_MS", "1000,5000,15000,60000").split(","))
CANCEL_ACK_LATENCY_MS = int(os.environ.get("MODL_EXECUTION_CANCEL_ACK_MS", "100"))
QUEUE_AHEAD_FRACTIONS = tuple(float(value) for value in os.environ.get("MODL_EXECUTION_QUEUE_FRACTIONS", "0,0.5,1.0").split(","))
MAKER_REBATE_GRID_BPS = tuple(float(value) for value in os.environ.get("MODL_EXECUTION_REBATE_GRID_BPS", "0,2.5").split(","))
BASE_LATENCY_MS = int(os.environ.get("MODL_EXECUTION_BASE_LATENCY_MS", "200"))
BASE_CANCEL_AFTER_MS = int(os.environ.get("MODL_EXECUTION_BASE_CANCEL_AFTER_MS", "15000"))
BASE_QUEUE_AHEAD_FRACTION = float(os.environ.get("MODL_EXECUTION_BASE_QUEUE_FRACTION", "1.0"))
BASE_MAKER_REBATE_BPS = float(os.environ.get("MODL_EXECUTION_BASE_REBATE_BPS", "2.5"))
SAVE_OUTPUTS = False

BAR_MS = BAR_MINUTES * 60_000
MARKOUT_MS = MARKOUT_MINUTES * 60_000

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 240)
pd.set_option("display.max_colwidth", 220)

ROOT, BAR_MINUTES, MARKOUT_MINUTES, ORDER_QTY

## Event Data Inventory

We use `received_mts` as the replay clock because that is the timestamp the local collector could have acted on. Exchange timestamps are useful diagnostics, but production cannot trade on data before it is received locally.

In [ ]:
def date_from_path(path: Path) -> str:
    tag = path.stem.rsplit("_", 1)[1]
    return datetime.strptime(tag, "%y-%m-%d").strftime("%Y-%m-%d")


def parquet_inventory(paths: list[Path], dataset: str) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for path in paths:
        lf = pl.scan_parquet(path)
        schema = lf.collect_schema()
        cols = schema.names()
        ts_col = "received_mts" if "received_mts" in cols else cols[0]
        stats = lf.select(
            pl.len().alias("rows"),
            pl.col(ts_col).cast(pl.Int64, strict=False).min().alias("min_mts"),
            pl.col(ts_col).cast(pl.Int64, strict=False).max().alias("max_mts"),
        ).collect().to_dicts()[0]
        rows.append(
            {
                "dataset": dataset,
                "date": date_from_path(path),
                "path": str(path),
                "columns": cols,
                **stats,
            }
        )
    return pd.DataFrame(rows)


hibachi_root = ROOT / "hibachi" / "btc_usdt-p"
quote_paths = sorted((hibachi_root / "quotes").glob("*.parquet"))
trade_paths = sorted((hibachi_root / "trades").glob("*.parquet"))
orderbook_paths = sorted((hibachi_root / "orderbook").glob("*.parquet"))

execution_inventory = pd.concat(
    [
        parquet_inventory(quote_paths, "hibachi_quotes"),
        parquet_inventory(trade_paths, "hibachi_trades"),
        parquet_inventory(orderbook_paths, "hibachi_orderbook_l2"),
    ],
    ignore_index=True,
)
execution_inventory["min_time"] = pd.to_datetime(execution_inventory["min_mts"], unit="ms", utc=True)
execution_inventory["max_time"] = pd.to_datetime(execution_inventory["max_mts"], unit="ms", utc=True)

raw_inventory = pd.DataFrame(
    [
        {"raw_file": str(path), "bytes": path.stat().st_size}
        for path in sorted((RAW_ROOT / "hibachi" / "btc_usdt-p").glob("**/*.jsonl.zst"))
    ]
)

display(execution_inventory[["dataset", "date", "rows", "min_time", "max_time"]])
display(raw_inventory.head(20))

In [ ]:
def load_hibachi_quotes(paths: list[Path]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for path in paths:
        frame = (
            pl.scan_parquet(path)
            .select(
                [
                    pl.col("received_mts").cast(pl.Int64),
                    pl.col("bid_price").cast(pl.Float64, strict=False).alias("bid"),
                    pl.col("ask_price").cast(pl.Float64, strict=False).alias("ask"),
                    pl.col("bid_size").cast(pl.Float64, strict=False).alias("bid_size"),
                    pl.col("ask_size").cast(pl.Float64, strict=False).alias("ask_size"),
                ]
            )
            .collect()
            .to_pandas()
        )
        frame["date"] = date_from_path(path)
        frames.append(frame)
    quotes = pd.concat(frames, ignore_index=True)
    quotes = quotes.dropna(subset=["received_mts", "bid", "ask", "bid_size", "ask_size"])
    quotes = quotes.sort_values("received_mts").drop_duplicates("received_mts", keep="last").reset_index(drop=True)
    quotes["mid"] = (quotes["bid"] + quotes["ask"]) / 2.0
    quotes["spread"] = quotes["ask"] - quotes["bid"]
    quotes["spread_bps"] = quotes["spread"] / quotes["mid"] * 10_000.0
    quotes["quote_time"] = pd.to_datetime(quotes["received_mts"], unit="ms", utc=True)
    return quotes


def load_hibachi_trades(paths: list[Path]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for path in paths:
        frame = (
            pl.scan_parquet(path)
            .select(
                [
                    pl.col("received_mts").cast(pl.Int64),
                    pl.col("trade_timestamp").cast(pl.Int64, strict=False),
                    pl.col("taker_side").cast(pl.Utf8).str.to_lowercase().alias("taker_side"),
                    pl.col("price").cast(pl.Float64, strict=False).alias("price"),
                    pl.col("quantity").cast(pl.Float64, strict=False).alias("qty"),
                ]
            )
            .collect()
            .to_pandas()
        )
        frame["date"] = date_from_path(path)
        frames.append(frame)
    trades = pd.concat(frames, ignore_index=True)
    trades = trades.dropna(subset=["received_mts", "price", "qty", "taker_side"])
    trades = trades.sort_values("received_mts").reset_index(drop=True)
    trades["trade_time"] = pd.to_datetime(trades["received_mts"], unit="ms", utc=True)
    raw_lag = trades["received_mts"] - trades["trade_timestamp"]
    trades["receive_lag_ms"] = raw_lag.where(raw_lag.abs() <= 60_000)
    trades["trade_timestamp_clock_usable"] = trades["receive_lag_ms"].notna()
    return trades


quotes = load_hibachi_quotes(quote_paths)
trades = load_hibachi_trades(trade_paths)

quote_gaps = quotes["received_mts"].diff().dropna()
trade_lag = trades["receive_lag_ms"].replace([np.inf, -np.inf], np.nan).dropna()
data_quality = pd.DataFrame(
    [
        {
            "quote_rows": len(quotes),
            "trade_rows": len(trades),
            "first_quote": quotes["quote_time"].min(),
            "last_quote": quotes["quote_time"].max(),
            "median_quote_gap_ms": quote_gaps.median(),
            "p95_quote_gap_ms": quote_gaps.quantile(0.95),
            "median_spread_bps": quotes["spread_bps"].median(),
            "p95_spread_bps": quotes["spread_bps"].quantile(0.95),
            "usable_trade_timestamp_share": trades["trade_timestamp_clock_usable"].mean(),
            "median_trade_receive_lag_ms": trade_lag.median() if not trade_lag.empty else np.nan,
            "p95_trade_receive_lag_ms": trade_lag.quantile(0.95) if not trade_lag.empty else np.nan,
        }
    ]
)
display(data_quality)
display(quotes.head())
display(trades.head())

## Decision Set

This notebook does not assume alpha is known. It creates a neutral execution test by joining both best bid and best ask at each bar close, then measuring what would happen under different latency, TTL, queue, and rebate assumptions.

This is the execution layer equivalent of a unit test: if joining the best price under realistic queue assumptions behaves very differently from cross-fill simulation, the alpha backtest must incorporate that difference.

In [ ]:
decision_quotes = quotes.copy()
decision_quotes["bar_end_mts"] = (decision_quotes["received_mts"] // BAR_MS) * BAR_MS + BAR_MS
decision_quotes = decision_quotes.groupby("bar_end_mts", as_index=False).tail(1).copy()
decision_quotes = decision_quotes[decision_quotes["bar_end_mts"] + MARKOUT_MS <= quotes["received_mts"].max()].copy()
decision_quotes["bar_end_time"] = pd.to_datetime(decision_quotes["bar_end_mts"], unit="ms", utc=True)

decision_summary = pd.DataFrame(
    [
        {
            "decision_bars": len(decision_quotes),
            "candidate_orders_per_scenario": len(decision_quotes) * 2,
            "first_decision": decision_quotes["bar_end_time"].min(),
            "last_decision": decision_quotes["bar_end_time"].max(),
            "median_decision_spread_bps": decision_quotes["spread_bps"].median(),
            "p95_decision_spread_bps": decision_quotes["spread_bps"].quantile(0.95),
        }
    ]
)
display(decision_summary)
decision_quotes.tail(10)

## Replay Primitives

The queue-adjusted fill rule is intentionally conservative for a join-best order. With `queue_ahead_fraction=1.0`, the full displayed top-of-book size must trade before our order is fully filled. With `0.0`, the trade-through fill is optimistic and assumes no queue ahead.

In [ ]:
quote_ts = quotes["received_mts"].to_numpy(dtype=np.int64)
quote_bid = quotes["bid"].to_numpy(dtype=float)
quote_ask = quotes["ask"].to_numpy(dtype=float)
quote_bid_size = quotes["bid_size"].to_numpy(dtype=float)
quote_ask_size = quotes["ask_size"].to_numpy(dtype=float)
quote_mid = quotes["mid"].to_numpy(dtype=float)

trade_ts = trades["received_mts"].to_numpy(dtype=np.int64)
trade_side = trades["taker_side"].astype(str).to_numpy()
trade_price = trades["price"].to_numpy(dtype=float)
trade_qty = trades["qty"].to_numpy(dtype=float)


def asof_quote_index(ts: int) -> int:
    idx = int(np.searchsorted(quote_ts, ts, side="right") - 1)
    if idx < 0 or idx >= len(quote_ts):
        return -1
    return idx


def first_quote_cross(side: str, price: float, start_mts: int, end_mts: int) -> float:
    left = int(np.searchsorted(quote_ts, start_mts, side="left"))
    right = int(np.searchsorted(quote_ts, end_mts, side="right"))
    if left >= right:
        return np.nan
    if side == "bid":
        condition = quote_ask[left:right] <= price
    else:
        condition = quote_bid[left:right] >= price
    hits = np.flatnonzero(condition)
    return float(quote_ts[left + hits[0]]) if len(hits) else np.nan


def first_trade_fill(
    side: str,
    price: float,
    start_mts: int,
    end_mts: int,
    queue_ahead_qty: float,
    order_qty: float = ORDER_QTY,
) -> tuple[float, float]:
    left = int(np.searchsorted(trade_ts, start_mts, side="left"))
    right = int(np.searchsorted(trade_ts, end_mts, side="right"))
    if left >= right:
        return np.nan, 0.0
    if side == "bid":
        condition = (trade_side[left:right] == "sell") & (trade_price[left:right] <= price)
    else:
        condition = (trade_side[left:right] == "buy") & (trade_price[left:right] >= price)
    candidate_offsets = np.flatnonzero(condition)
    if not len(candidate_offsets):
        return np.nan, 0.0
    cumulative_qty = 0.0
    last_trade_mts = np.nan
    for offset in candidate_offsets:
        cumulative_qty += trade_qty[left + offset]
        last_trade_mts = float(trade_ts[left + offset])
        filled_qty = max(0.0, min(order_qty, cumulative_qty - queue_ahead_qty))
        if filled_qty >= order_qty:
            return last_trade_mts, filled_qty
    partial_qty = max(0.0, min(order_qty, cumulative_qty - queue_ahead_qty))
    if partial_qty <= 0:
        return np.nan, 0.0
    return last_trade_mts, partial_qty


def markout_bps(side: str, fill_price: float, fill_mts: float) -> float:
    idx = asof_quote_index(int(fill_mts + MARKOUT_MS))
    if idx < 0:
        return np.nan
    if side == "bid":
        return float(10_000.0 * np.log(quote_mid[idx] / fill_price))
    return float(10_000.0 * np.log(fill_price / quote_mid[idx]))


def simulate_execution_scenario(
    latency_ms: int,
    cancel_after_ms: int,
    queue_ahead_fraction: float,
    maker_rebate_bps: float,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for decision in decision_quotes.itertuples(index=False):
        for side in ("bid", "ask"):
            live_mts = int(decision.bar_end_mts + latency_ms)
            quote_idx = asof_quote_index(live_mts)
            if quote_idx < 0:
                continue
            if side == "bid":
                order_price = float(quote_bid[quote_idx])
                visible_queue = float(quote_bid_size[quote_idx])
            else:
                order_price = float(quote_ask[quote_idx])
                visible_queue = float(quote_ask_size[quote_idx])
            cancel_effective_mts = live_mts + cancel_after_ms + CANCEL_ACK_LATENCY_MS
            queue_ahead_qty = max(0.0, visible_queue * queue_ahead_fraction)

            cross_fill_mts = first_quote_cross(side, order_price, live_mts, cancel_effective_mts)
            trade_fill_mts, trade_fill_qty = first_trade_fill(side, order_price, live_mts, cancel_effective_mts, queue_ahead_qty=0.0)
            queue_fill_mts, queue_fill_qty = first_trade_fill(side, order_price, live_mts, cancel_effective_mts, queue_ahead_qty=queue_ahead_qty)
            full_queue_fill = bool(np.isfinite(queue_fill_mts) and queue_fill_qty >= ORDER_QTY)
            markout = markout_bps(side, order_price, queue_fill_mts) if full_queue_fill else np.nan
            value = markout + maker_rebate_bps if np.isfinite(markout) else 0.0

            rows.append(
                {
                    "bar_end_mts": int(decision.bar_end_mts),
                    "bar_end_time": decision.bar_end_time,
                    "side": side,
                    "latency_ms": latency_ms,
                    "cancel_after_ms": cancel_after_ms,
                    "cancel_ack_latency_ms": CANCEL_ACK_LATENCY_MS,
                    "queue_ahead_fraction": queue_ahead_fraction,
                    "maker_rebate_bps": maker_rebate_bps,
                    "live_mts": live_mts,
                    "order_price": order_price,
                    "visible_queue_qty": visible_queue,
                    "queue_ahead_qty": queue_ahead_qty,
                    "cross_fill": np.isfinite(cross_fill_mts),
                    "trade_fill": np.isfinite(trade_fill_mts),
                    "queue_fill": full_queue_fill,
                    "partial_fill_qty": queue_fill_qty,
                    "cross_fill_mts": cross_fill_mts,
                    "trade_fill_mts": trade_fill_mts,
                    "queue_fill_mts": queue_fill_mts,
                    "queue_fill_delay_ms": queue_fill_mts - live_mts if np.isfinite(queue_fill_mts) else np.nan,
                    "markout_bps": markout,
                    "posted_value_bps": value,
                }
            )
    return pd.DataFrame(rows)


def summarize_orders(orders: pd.DataFrame) -> dict[str, float]:
    filled = orders[orders["queue_fill"]]
    mean_markout = filled["markout_bps"].mean() if len(filled) else np.nan
    return {
        "orders": int(len(orders)),
        "cross_fill_rate": float(orders["cross_fill"].mean()) if len(orders) else np.nan,
        "trade_fill_rate_no_queue": float(orders["trade_fill"].mean()) if len(orders) else np.nan,
        "queue_full_fill_rate": float(orders["queue_fill"].mean()) if len(orders) else np.nan,
        "partial_fill_rate": float((orders["partial_fill_qty"] > 0).mean()) if len(orders) else np.nan,
        "full_fills": int(orders["queue_fill"].sum()) if len(orders) else 0,
        "mean_markout_if_full_bps": float(mean_markout) if np.isfinite(mean_markout) else np.nan,
        "required_rebate_to_zero_per_fill_bps": float(max(0.0, -mean_markout)) if np.isfinite(mean_markout) else np.nan,
        "posted_ev_before_rebate_bps": float(np.where(orders["queue_fill"], orders["markout_bps"], 0.0).mean()) if len(orders) else np.nan,
        "posted_ev_after_rebate_bps": float(orders["posted_value_bps"].mean()) if len(orders) else np.nan,
        "median_fill_delay_ms": float(filled["queue_fill_delay_ms"].median()) if len(filled) else np.nan,
    }

## Base Scenario

The base scenario uses a 200 ms decision-to-live delay, 15 second TTL, 100 ms cancel acknowledgement latency, full visible top-of-book queue ahead, and a 2.5 bps maker rebate.

In [ ]:
base_orders = simulate_execution_scenario(
    latency_ms=BASE_LATENCY_MS,
    cancel_after_ms=BASE_CANCEL_AFTER_MS,
    queue_ahead_fraction=BASE_QUEUE_AHEAD_FRACTION,
    maker_rebate_bps=BASE_MAKER_REBATE_BPS,
)
base_summary = pd.DataFrame([summarize_orders(base_orders)])
base_by_side = base_orders.groupby("side").apply(lambda frame: pd.Series(summarize_orders(frame)), include_groups=False).reset_index()

display(base_summary)
display(base_by_side)
base_orders.tail(20)

## Cross-Fill Versus Queue-Fill Gap

This table is the direct live-vs-sim warning. If `cross_only` is large, a bar/cross simulator is filling orders that may not survive queue assumptions. If `queue_without_cross` is large, a cross-only simulator is missing fills that can happen when aggressive flow hits us without the quote crossing.

In [ ]:
fill_gap = base_orders.copy()
fill_gap["fill_relation"] = np.select(
    [
        fill_gap["cross_fill"] & fill_gap["queue_fill"],
        fill_gap["cross_fill"] & ~fill_gap["queue_fill"],
        ~fill_gap["cross_fill"] & fill_gap["queue_fill"],
    ],
    ["both", "cross_only", "queue_without_cross"],
    default="neither",
)
fill_gap_summary = (
    fill_gap.groupby(["side", "fill_relation"], as_index=False)
    .agg(
        orders=("fill_relation", "count"),
        mean_markout_bps=("markout_bps", "mean"),
        posted_ev_after_rebate_bps=("posted_value_bps", "mean"),
    )
)
fill_gap_summary["share"] = fill_gap_summary["orders"] / len(base_orders)
display(fill_gap_summary.sort_values(["side", "fill_relation"]))

## Latency, Cancel, Queue, And Rebate Grid

The grid shows what we can change operationally:

1. lower forecast/order latency,
2. shorter or longer quote TTL,
3. better queue position,
4. better maker fee tier.

The best scenario is not necessarily the one with the highest fill rate. Toxic fills can make more fills worse.

In [ ]:
scenario_rows: list[dict[str, object]] = []
scenario_samples: list[pd.DataFrame] = []

for latency_ms in FORECAST_AND_SEND_LATENCY_MS:
    for cancel_after_ms in CANCEL_AFTER_MS:
        for queue_fraction in QUEUE_AHEAD_FRACTIONS:
            for maker_rebate_bps in MAKER_REBATE_GRID_BPS:
                orders = simulate_execution_scenario(
                    latency_ms=latency_ms,
                    cancel_after_ms=cancel_after_ms,
                    queue_ahead_fraction=queue_fraction,
                    maker_rebate_bps=maker_rebate_bps,
                )
                row = {
                    "latency_ms": latency_ms,
                    "cancel_after_ms": cancel_after_ms,
                    "queue_ahead_fraction": queue_fraction,
                    "maker_rebate_bps": maker_rebate_bps,
                    **summarize_orders(orders),
                }
                scenario_rows.append(row)
                if latency_ms == BASE_LATENCY_MS and maker_rebate_bps == BASE_MAKER_REBATE_BPS:
                    scenario_samples.append(orders)

scenario_summary = pd.DataFrame(scenario_rows).sort_values("posted_ev_after_rebate_bps", ascending=False).reset_index(drop=True)
display(scenario_summary.head(30))

base_rebate_scenarios = scenario_summary[scenario_summary["maker_rebate_bps"] == BASE_MAKER_REBATE_BPS].copy()
display(base_rebate_scenarios.sort_values("posted_ev_after_rebate_bps", ascending=False).head(30))

## Side-Level Grid

A strategy may not need symmetric quoting. If bid and ask execution quality differs, the inventory model should quote asymmetrically rather than forcing both sides.

In [ ]:
side_rows: list[dict[str, object]] = []
for latency_ms in FORECAST_AND_SEND_LATENCY_MS:
    for cancel_after_ms in CANCEL_AFTER_MS:
        for queue_fraction in QUEUE_AHEAD_FRACTIONS:
            orders = simulate_execution_scenario(
                latency_ms=latency_ms,
                cancel_after_ms=cancel_after_ms,
                queue_ahead_fraction=queue_fraction,
                maker_rebate_bps=BASE_MAKER_REBATE_BPS,
            )
            for side, frame in orders.groupby("side"):
                side_rows.append(
                    {
                        "side": side,
                        "latency_ms": latency_ms,
                        "cancel_after_ms": cancel_after_ms,
                        "queue_ahead_fraction": queue_fraction,
                        **summarize_orders(frame),
                    }
                )
side_scenario_summary = pd.DataFrame(side_rows).sort_values("posted_ev_after_rebate_bps", ascending=False).reset_index(drop=True)
display(side_scenario_summary.head(40))

## Diagnostics

These charts show how execution EV changes with TTL, latency, queue position, and maker rebate.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plot_grid = scenario_summary[scenario_summary["maker_rebate_bps"] == BASE_MAKER_REBATE_BPS].copy()
sns.lineplot(data=plot_grid, x="cancel_after_ms", y="posted_ev_after_rebate_bps", hue="queue_ahead_fraction", style="latency_ms", marker="o", ax=axes[0, 0])
axes[0, 0].axhline(0, color="black", linewidth=1)
axes[0, 0].set_xscale("log")
axes[0, 0].set_title("Posted EV After Rebate")

sns.lineplot(data=plot_grid, x="cancel_after_ms", y="queue_full_fill_rate", hue="queue_ahead_fraction", style="latency_ms", marker="o", ax=axes[0, 1])
axes[0, 1].set_xscale("log")
axes[0, 1].set_title("Full Fill Rate")

sns.scatterplot(data=scenario_summary, x="queue_full_fill_rate", y="mean_markout_if_full_bps", hue="maker_rebate_bps", size="cancel_after_ms", ax=axes[1, 0])
axes[1, 0].axhline(0, color="black", linewidth=1)
axes[1, 0].set_title("Fill Rate Versus Markout")

base_heat = plot_grid[plot_grid["queue_ahead_fraction"] == BASE_QUEUE_AHEAD_FRACTION].pivot(index="latency_ms", columns="cancel_after_ms", values="posted_ev_after_rebate_bps")
sns.heatmap(base_heat, annot=True, fmt=".2f", center=0, cmap="vlag", ax=axes[1, 1])
axes[1, 1].set_title("EV Heatmap, Full Queue Ahead")

plt.tight_layout()

plt.figure(figsize=(12, 5))
sns.barplot(data=fill_gap_summary, x="fill_relation", y="orders", hue="side")
plt.title("Base Scenario: Cross Fill Versus Queue Fill")
plt.tight_layout()

## Production Validation Requirements

The event replay gets us closer than bar-level labels, but the live validation stack still needs production instrumentation.

Minimum records to log for sim/paper/live comparison:

1. `decision_id`, bar close timestamp, forecast start/end timestamp, model version, feature snapshot hash.
2. local market-data receive timestamp, exchange timestamp if present, connection id, and raw message offset or file position.
3. order intent: side, price, quantity, post-only flag, TTL, cancel condition, inventory at decision.
4. order lifecycle: submit local time, exchange ack time, live order id, reject reason, cancel submit time, cancel ack time.
5. fills: fill local time, exchange fill time, side, price, quantity, fee/rebate, liquidity flag.
6. replay inputs: the exact raw market-data capture around each decision window.

The critical test is not whether historical simulation is positive. The critical test is whether `sim`, `paper`, and `live` share the same discrete decisions and whether live fill quality matches replay after latency and queue assumptions. Full L2 depth reconstruction is the next extension for quotes away from the best price.

## Portfolio Manager Interpretation

What can make standalone passive execution positive?

1. **Queue position**: reducing queue ahead is the biggest lever. Being first at the level is a different strategy from joining behind visible size.

2. **Cancel discipline**: short TTL can improve markout by avoiding stale toxic quotes, but it reduces fill count and capacity.

3. **Latency**: lower latency helps if it lets us post or cancel before adverse flow arrives. It is not automatically good or bad; it changes the fill set.

4. **Rebates**: a `2.5 bps` maker rebate is meaningful, especially for near-flat markout scenarios. It does not rescue fills with `-8` to `-30 bps` adverse markouts.

5. **Alpha selection**: execution replay should be run on actual strategy decisions next, not neutral both-side decisions. That is where we test whether alpha picks the fills with good markouts or accidentally selects toxic fills.

The next notebook should feed actual quote decisions from `inventory_quote_model.ipynb` into this replay rather than quoting both sides mechanically.

## Optional Save

Set `SAVE_OUTPUTS = True` to write event inventory, base replay, fill-gap, and scenario-grid tables.

In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / "execution_replay" / f"bar_{BAR_MINUTES}m"
    out_dir.mkdir(parents=True, exist_ok=True)
    execution_inventory.to_parquet(out_dir / "execution_inventory.parquet")
    data_quality.to_parquet(out_dir / "data_quality.parquet")
    decision_summary.to_parquet(out_dir / "decision_summary.parquet")
    base_orders.to_parquet(out_dir / "base_orders.parquet")
    base_summary.to_parquet(out_dir / "base_summary.parquet")
    fill_gap_summary.to_parquet(out_dir / "fill_gap_summary.parquet")
    scenario_summary.to_parquet(out_dir / "scenario_summary.parquet")
    side_scenario_summary.to_parquet(out_dir / "side_scenario_summary.parquet")
    print(f"wrote execution replay outputs to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; nothing written")